# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected. Tables ready.")

Connected. Tables ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content item (`content_hash_id`) belonging to
one client (`client_hash_id`), summarizing its daily search performance aggregated
over a defined 90-day window. The raw source table (`fact_content_daily_performance`)
is grain = one row per day × client × content item; my feature table aggregates
that up to one row per content item per window.

**Time window:** I'll iterate using a mid-panel month, `month=2026-03`, as instructed —
NOT the June 2026 `_sample` table, since that's the sealed final month reserved
for outcome testing, not feature development.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** prior-30-day impressions, prior-30-day clicks, average position,
CTR, content age — all knowable before the decision point.

**Label/proxy:** `is_declining` = impressions in the most recent 30-day sub-window
fell more than 20% versus the prior 30-day sub-window. This is an observed proxy,
not a causal outcome.

**Context (not features, not labels):** `dim_clients.gsc_data_start` /
`ga4_data_start` — used only to check data availability per client, not fed to the model.

**Excluded (and why):** `fact_content_query_90d` (query-mix signals) — deliberately
excluded from this first contract. It adds real complexity (query diversity,
concentration) that isn't needed to prove the core refresh-scoring pattern yet;
I may add it in a later modeling week once the simpler version is validated.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1 — Grain check:

In [2]:
grain_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  distinct_grain
0     9841378         9841378


If total_rows equals distinct_grain, your claimed grain (one row per day×client×content) is confirmed — no duplicates.


Query 2 — Row count + date span of your slice:

In [3]:
slice_stats = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print(slice_stats)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


Query 3 — Availability check with IS TRUE:

In [4]:
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows,
           COUNT(*) FILTER (WHERE gsc_impressions IS NOT NULL) AS gsc_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_rows  gsc_rows
0     9841378    413966   9841378


(If ga4_data_available isn't the exact column name in your version, run DESCRIBE {TABLES['fact_daily']} first to check real column names — I'm working from the guide's description, not a live schema, so verify this before trusting it.)

Five features + the leakage trap (code cell)

In [5]:
features = con.sql(f"""
    WITH bounds AS (SELECT DATE '2026-04-01' AS cutoff),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.cutoff - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.cutoff - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               AVG(f.gsc_avg_position) AS avg_position,
               AVG(f.gsc_clicks * 1.0 / NULLIF(f.gsc_impressions,0)) AS ctr
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.cutoff - INTERVAL 60 DAY AND f.report_date <= b.cutoff
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

features['is_declining'] = (features['imp_last30'] < 0.8 * features['imp_prev30']).astype(int)
print(f"{len(features):,} rows")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

82,522 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,avg_position,ctr,is_declining
0,client_e547b89c05043229,content_7995404695ee1ffd,759.0,1046.0,32.236685,0.002367,1
1,client_e547b89c05043229,content_ccbb253f142217c3,3090.0,1719.0,21.226886,0.003117,0
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,544.0,889.0,6.688850,0.000000,1
3,client_e547b89c05043229,content_acf700633f016e5a,213.0,252.0,6.864457,0.000000,0
4,client_e547b89c05043229,content_712e44562fed8ff2,510.0,324.0,7.511298,0.000466,0


1. `imp_prev30` — knowable because it's from the window strictly before the decision point.
2. `avg_position` — knowable, it's an observed historical average, not a future value.
3. `ctr` — knowable, calculated from past clicks/impressions only.
4. Content age (would add: `content_age_days`) — knowable, static/historical fact.
5. Client tenure (`gsc_data_start`) — knowable, fixed metadata about the client.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

# Deliberately leaky: imp_last30 is literally what the label is computed from
X = features[['imp_prev30', 'avg_position', 'ctr', 'imp_last30']].fillna(0)  # <-- leak
y = features['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
print("WITH LEAK — precision:", precision_score(y_te, model.predict(X_te)))

WITH LEAK — precision: 0.9905944986690328


In [7]:
# Now remove the leak and re-check the honest number
X_honest = features[['imp_prev30', 'avg_position', 'ctr']].fillna(0)
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
model_honest = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
print("HONEST — precision:", precision_score(y_te, model_honest.predict(X_te)))

HONEST — precision: 0.409778812572759


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data cannot tell me *why* a page is declining (consolidation, seasonality,
SERP changes, or a genuine content problem all look similar without more checks).
The panel is also unbalanced — clients have different history lengths, and early
rows for newer clients are GSC-only (no GA4/session data), so any window I define
must be checked against `dim_clients.gsc_data_start`/`ga4_data_start` first, or I
risk mistaking "no tracking yet" for "no traffic."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.